In [2]:
import pandas as pd

/Users/maryamzakiyya/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
df = pd.read_csv('all_files.csv')
df

,Unnamed: 0,file_name,folder_name,submissions,weight,text
0,0,11-13-20 Khan.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Hareem Khan Sent: Thursday, November 12,..."
1,1,11-13-20 Solkovits.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: gs <gsolk@aol.com> Sent: Thursday, Novem..."
2,2,11-13-20 Kaur and Singh.pdf,"Comments Received After September 30, 2020",1,0.000008,"November 13th, 2020\n\nDear Superintendent Thu..."
3,3,11-19-20 Lamont.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sami Lamont Sent: Wednesday, November 18..."
4,4,11-18-20 Jensen.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sine Hwang Jensen Sent: Tuesday, Novembe..."
...,...,...,...,...,...,...
7712,7713,10-27-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."
7714,7715,10-22-20 Nash.pdf,"Comments Received After September 30, 2020",1,0.000008,"Amin Nash, M.A. 10.21.2020\n\n1\n\nTo Executiv..."
7715,7716,10-27-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 27, ..."


In [4]:
import pandas as pd

# Example DataFrame

# Check which rows are empty or just spaces
empty_mask = df['text'].str.strip().eq('') | df['text'].isna()

df[empty_mask]


,Unnamed: 0,file_name,folder_name,submissions,weight,text


In [5]:
import string
def is_illegible(s):
    if pd.isna(s) or s.strip() == '':
        return False  # treat empty as separate case
    return any(c not in string.printable for c in s)


illegible_rows = df[df['text'].apply(is_illegible)]

illegible_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
5,5,11-17-20 Yardeni.pdf,"Comments Received After September 30, 2020",1,0.000008,----------------------------------------------...
13,13,11-13-20 Epstein et al.pdf,"Comments Received After September 30, 2020",1,0.000008,Dear Members of the Instructional Quality Comm...
15,15,11-13-2- Khalili.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Zoha Khalili Sent: Friday, November 13, ..."
16,16,11-11-20 DePass.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Linval DePass Sent: Wednesday, November ..."
17,17,11-13-20 Homsi.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Pauline Homsi Sent: Friday, November 13,..."
...,...,...,...,...,...,...
7709,7710,10-2-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 2, 20..."
7710,7711,10-2-20 Boyd.pdf,"Comments Received After September 30, 2020",1,0.000008,"October 2, 2020\nPresident\nE. Toby Boyd\n\nTO..."
7711,7712,10-14-20 George.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: IQC\nSent: Wednesday, October 14, 2020 1..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."


In [6]:
import re
import pandas as pd

def legibility_metrics(s):
    if pd.isna(s) or not isinstance(s, str) or s.strip() == '':
        return {"alpha_ratio": 0, "word_ratio": 0, "weird_char_ratio": 0}
    
    s_clean = s.replace('\n', ' ').replace('\r', ' ')
    total_chars = len(s_clean)
    alpha_chars = sum(c.isalpha() for c in s_clean)
    alpha_ratio = alpha_chars / total_chars if total_chars > 0 else 0

    words = re.findall(r'[A-Za-z]{2,}', s_clean)
    word_ratio = len(words) / (total_chars / 5) if total_chars > 0 else 0

    weird_char_ratio = sum(not (c.isalnum() or c.isspace()) for c in s_clean) / total_chars

    return {
        "alpha_ratio": alpha_ratio,
        "word_ratio": word_ratio,
        "weird_char_ratio": weird_char_ratio
    }

# Apply and add to df
metrics = df['text'].apply(legibility_metrics).apply(pd.Series)
df_metrics = pd.concat([df, metrics], axis=1)


In [7]:
df_metrics[['alpha_ratio', 'word_ratio', 'weird_char_ratio']].describe()

,alpha_ratio,word_ratio,weird_char_ratio
count,7717.000000,7717.000000,7717.000000
mean,0.790383,0.738118,0.030825
std,0.025821,0.044306,0.011637
min,0.333333,0.043838,0.000000
25%,0.783047,0.709681,0.024486
50%,0.795695,0.736152,0.028606
75%,0.805656,0.764563,0.033719
max,0.871165,0.948827,0.253937


In [8]:
alpha_low = df_metrics['alpha_ratio'].mean() - 3 * df_metrics['alpha_ratio'].std()   # unusually low alphabetic %
word_low = df_metrics['word_ratio'].mean() - 3 * df_metrics['word_ratio'].std()     # unusually low English word %
weird_high = df_metrics['weird_char_ratio'].mean() + 3 * df_metrics['weird_char_ratio'].std()  # unusually high symbol %

illegible_rows = df[
    (df_metrics['alpha_ratio'] < alpha_low) |
    (df_metrics['word_ratio'] < word_low) |
    (df_metrics['weird_char_ratio'] > weird_high)
]

illegible_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
5,5,11-17-20 Yardeni.pdf,"Comments Received After September 30, 2020",1,0.000008,----------------------------------------------...
13,13,11-13-20 Epstein et al.pdf,"Comments Received After September 30, 2020",1,0.000008,Dear Members of the Instructional Quality Comm...
86,86,11-11-20 Guttenberg.pdf,"Comments Received After September 30, 2020",1,0.000008,-----Original Message----From: martaguttenberg...
133,133,11-13-20 Silton.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: On Behalf Of Lynn Silton Sent: Friday, N..."
139,139,11-10-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Tuesday, November 10, ..."
...,...,...,...,...,...,...
7673,7674,10-27-20 Cohen.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ken Cohen\nSent: Monday, October 26, 202..."
7684,7685,11-2-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, November 2, 2..."
7691,7692,10-14-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 13, ..."
7692,7693,10-26-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."


In [9]:
# Recalculate new (looser) thresholds
alpha_low_2 = df_metrics['alpha_ratio'].mean() - 2 * df_metrics['alpha_ratio'].std()
word_low_2 = df_metrics['word_ratio'].mean() - 2 * df_metrics['word_ratio'].std()
weird_high_2 = df_metrics['weird_char_ratio'].mean() + 2 * df_metrics['weird_char_ratio'].std()

# Run again to find more rows that look illegible under looser rules
illegible_rows_looser = df[
    (df_metrics['alpha_ratio'] < alpha_low_2) |
    (df_metrics['word_ratio'] < word_low_2) |
    (df_metrics['weird_char_ratio'] > weird_high_2)
]

# Keep only new ones not in the first batch
new_illegible_rows = illegible_rows_looser.loc[
    ~illegible_rows_looser.index.isin(illegible_rows.index)
]

new_illegible_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
105,105,11-12-20 Kaufman.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Jon Kaufman Sent: Thursday, November 12,..."
175,175,11-18-20 Hsiao.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Rod Hsiao Sent: Wednesday, November 18, ..."
177,177,11-13-20 Baha.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Omar Baha Sent: Friday, November 13, 202..."
200,200,11-13-20 Webber Attachment 1.pdf,"Comments Received After September 30, 2020",1,0.000008,Arabic 367: American Identity in the World Ara...
250,250,11-13-20 Chang Attachment 2.pdf,"Comments Received After September 30, 2020",1,0.000008,1 This lesson was submitted by a member(s) of ...
...,...,...,...,...,...,...
7699,7700,10-13-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 13, ..."
7700,7701,10-1-20 Klein.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Steve Klein\nSent: Thursday, October 1, ..."
7705,7706,10-19-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 19, 2..."
7714,7715,10-22-20 Nash.pdf,"Comments Received After September 30, 2020",1,0.000008,"Amin Nash, M.A. 10.21.2020\n\n1\n\nTo Executiv..."


# Antisemitism Sampling

In [133]:
antisemitism_variants = [
    'antisemitism',
    'anti-semitism',
    'anti semitism',
    'antisemitic',
    'anti-semitic',
    'anti semitic',
    'antisemite',
    'anti-semite',
    'anti semite',
    'antisemitisim',
    'antisemitsm',
    'antisemitizm',
    'anti-semitizm',
    'anti semitizm',
    'ant1semitism',
    'anti$emitism',
    'anti-semit1sm',
    'anti semit1sm'
]

pattern = '|'.join(antisemitism_variants)

matched_rows = df[df['text'].str.contains(pattern, case=False, na=False)]

matched_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
0,0,11-13-20 Khan.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Hareem Khan Sent: Thursday, November 12,..."
1,1,11-13-20 Solkovits.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: gs <gsolk@aol.com> Sent: Thursday, Novem..."
5,5,11-17-20 Yardeni.pdf,"Comments Received After September 30, 2020",1,0.000008,----------------------------------------------...
8,8,11-16-20 Landau.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: nonielandau Sent: Friday, November 13, 2..."
9,9,11-18-20 Donsky.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Joanne Donsky Sent: Wednesday, November ..."
...,...,...,...,...,...,...
7707,7708,10-12-20 Haufrect.pdf,"Comments Received After September 30, 2020",1,0.000008,Public Input Template–2020 Ethnic Studies Mode...
7708,7709,10-30-20 Meyers and Shmueli 2.pdf,"Comments Received After September 30, 2020",1,0.000008,Educators for Excellence in Ethnic Studies\n\n...
7709,7710,10-2-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 2, 20..."
7711,7712,10-14-20 George.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: IQC\nSent: Wednesday, October 14, 2020 1..."


In [134]:
matched_rows['submissions'].sum()

20519

In [ ]:
antisemitism_samples = df.sample(n=40, replace=True, random_state=42)
antisemitism_samples

# Recurring Authors

In [22]:
df = pd.read_csv('all_files.csv')
df

,Unnamed: 0,file_name,folder_name,submissions,weight,text
0,0,11-13-20 Khan.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Hareem Khan Sent: Thursday, November 12,..."
1,1,11-13-20 Solkovits.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: gs <gsolk@aol.com> Sent: Thursday, Novem..."
2,2,11-13-20 Kaur and Singh.pdf,"Comments Received After September 30, 2020",1,0.000008,"November 13th, 2020\n\nDear Superintendent Thu..."
3,3,11-19-20 Lamont.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sami Lamont Sent: Wednesday, November 18..."
4,4,11-18-20 Jensen.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sine Hwang Jensen Sent: Tuesday, Novembe..."
...,...,...,...,...,...,...
7712,7713,10-27-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."
7714,7715,10-22-20 Nash.pdf,"Comments Received After September 30, 2020",1,0.000008,"Amin Nash, M.A. 10.21.2020\n\n1\n\nTo Executiv..."
7715,7716,10-27-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 27, ..."


In [23]:
authors = df['file_name'].str.extract(r'\d{1,2}-\d{1,2}-\d{2}\s+(.*?)\.pdf')

author_counts = authors[0].value_counts().reset_index()
author_counts.columns = ['author', 'count']

In [24]:
author_counts[author_counts['count'] > 1]

,author,count
0,Parker,46
1,Cohen,23
2,Smith,17
3,Schwartz,15
4,Levin,14
...,...,...
967,Herman,2
968,Bender,2
969,LeMay,2
970,Ellis,2


In [58]:
author_counts.head(70)

,author,count
0,Parker,46
1,Cohen,23
2,Smith,17
3,Schwartz,15
4,Levin,14
...,...,...
65,Ross,6
66,Clark,6
67,Freeman,6
68,Goldman,6


In [26]:
author_parker = df[df['file_name'].str.contains('Parker', case=False)]

In [27]:
author_parker[author_parker['text'].str.contains('Ruth', case=False)]
ruth_parker = author_parker.drop(['Unnamed: 0', 'submissions', 'weight'], axis=1)
ruth_parker

,file_name,folder_name,text
46,11-13-20 Parker Ella.pdf,"Comments Received After September 30, 2020","From: Ella Parker Sent: Friday, November 13, 2..."
54,11-12-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ella Parker Sent: Thursday, November 12,..."
61,11-29-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker Sent: Sunday, November 29, 2..."
75,11-13-20 Parker.pdf,"Comments Received After September 30, 2020","From: Maria Parker Sent: Friday, November 13, ..."
83,11-19-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker Sent: Thursday, November 19,..."
...,...,...,...
7706,10-12-20 Parker 2.pdf,"Comments Received After September 30, 2020","From: Ruth Parker\nSent: Monday, October 12, 2..."
7709,10-2-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker\nSent: Friday, October 2, 20..."
7712,10-27-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,10-30-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker\nSent: Friday, October 30, 2..."


In [28]:
ruth_parker.to_csv('ruth_parker.csv', index=False)

In [29]:
df[df['text'].str.contains('Ruth Parker', case=False)]

,Unnamed: 0,file_name,folder_name,submissions,weight,text
61,61,11-29-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Sunday, November 29, 2..."
83,83,11-19-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Thursday, November 19,..."
90,90,12-3-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Thursday, December 3, ..."
110,110,11-17-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Tuesday, November 17, ..."
139,139,11-10-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Tuesday, November 10, ..."
...,...,...,...,...,...,...
7709,7710,10-2-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 2, 20..."
7712,7713,10-27-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."
7715,7716,10-27-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 27, ..."


In [30]:
print(df.iloc[7716]['text'])

From: Sue Fishkoff
Sent: Monday, October 26, 2020 6:28 PM
To: Ruth Parker
Cc: [list of other recipients redacted]; Ethnic Studies
Subject: [EXTERNAL] Re: Fw: A forward from J. Barry's friend, Marilyn; how can any self-respecting
person trust the NYT after knowing this?
Ruth,
This is Sue Fishkoff here, editor of J. The Jewish News of Northern California. I'm glad I'm on your email
list, because I can allay your fears.
What you have sent us is a piece of satire created in 2006, to protest the Israeli operation in Lebanon.
The real May 10, 1943 front page of the New York Times is here (you have to scroll down past the fake
front page, the explanation, and then you get to the real front page, which you can verify on the New
York Times website itself):
https://www.snopes.com/fact-check/warsaw-ghetto-uprising/
It is awful to see this true "fake news." Whatever you think of the New York Times, it would never have
published such Nazi propaganda.
Regards,
Sue
On Mon, Oct 26, 2020 at 5:41 PM Rut

In [62]:
# 1. Extract authors from filenames and get top 100
author_counts = (
    df['file_name']
    .str.extract(r'\d{1,2}-\d{1,2}-\d{2}\s+(.*?)\.pdf')[0]
    .value_counts()
)

top_authors = author_counts.head(100).index

# 2. Extract all "From:" names for the top 100 authors
all_names = []

for author in top_authors:
    subset = df[df['file_name'].str.contains(author, case=False)].copy()
    names = subset['text'].str.extract(r'From:\s*(.*?)\s+Sent:', expand=False)
    all_names.append(names)

# 3. Combine and get top 10 names
top_15 = (
    pd.concat(all_names)
    .value_counts()
    .head(15)
)

top_15


text
Ruth Parker               78
Sondra Goldstein           9
Victoria Wong              7
JEFFREY CARMEL             6
Nora Rousso                6
[email redacted]           6
                           6
Joanne Donsky              6
Joe Nalven                 5
Kim Green                  5
Cynthia Chang              5
Lee, Roselinn ( Linn )     4
Kathy Jordan               4
Linda Brownstein           4
HELEN WEINSTEIN            4
Name: count, dtype: int64

# Mentions of BDS

In [135]:
bds_variants = [
    'bds',
    'B.D.S.',
    'Boycott, Divestment, and Sanctions',
    'Boycott, Divestment, and Sanction',
    'Boycott & Divestment',
    'Boycott and Divestment'
]

pattern = '|'.join(bds_variants)

matched_rows = df[df['text'].str.contains(pattern, case=False, na=False)]
matched_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
38,38,11-23-20 Goldfinger.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Robin Goldfinger Sent: Saturday, Novembe..."
52,52,11-18-20 Penhaskashi.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: JADEN PENHASKASHI Sent: Wednesday, Novem..."
58,58,11-12-20 Kay.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sheila Kay Sent: Thursday, November 12, ..."
70,70,"11-08-20 Smith, Gregory.pdf","Comments Received After September 30, 2020",1,0.000008,"From: Greg Smith Sent: Sunday, November 8, 202..."
74,74,11-13-20 Paperman.pdf,"Comments Received After September 30, 2020",1,0.000008,Debbie Paperman Board Member Democrats for Isr...
...,...,...,...,...,...,...
7667,7668,11-4-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Wednesday, November 4..."
7679,7680,10-5-20 Chiu.pdf,"Comments Received After September 30, 2020",1,0.000008,"September 30, 2020\nInstructional Quality Comm..."
7691,7692,10-14-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 13, ..."
7707,7708,10-12-20 Haufrect.pdf,"Comments Received After September 30, 2020",1,0.000008,Public Input Template–2020 Ethnic Studies Mode...


In [136]:
matched_rows['submissions'].sum()

7402

# Mentions of Nakba

In [137]:
nakba_variants = [
    'nakba',
    'Nakbah',
    '1948',
    'War of Independence',
    'Arab–Israeli War'
]

pattern = '|'.join(nakba_variants)

matched_rows = df[df['text'].str.contains(pattern, case=False, na=False)]
matched_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
20,20,11-18-20 Smith.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Warren Smith Sent: Wednesday, November 1..."
41,41,11-12-20 Galal.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: KGalal Sent: Thursday, November 12, 2020..."
79,79,11-12-20 Lepawsky.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Jed Linden Sent: Thursday, November 12, ..."
108,108,11-13-20 Lavie.pdf,"Comments Received After September 30, 2020",1,0.000008,"Smadar Lavie, PhD Professor Emerita of Anthrop..."
114,114,11-13-20 Shoman Attachment.pdf,"Comments Received After September 30, 2020",1,0.000008,Arab Americans\n\nSample Lesson: Understanding...
...,...,...,...,...,...,...
7622,7623,Pietruszka.pdf,Comments Received After Field Review,1,0.000017,Dear CA Department of Education Ethnic Studies...
7628,7629,Bertet.pdf,Comments Received After Field Review,1,0.000017,"Julian Bertet\n4121 Vantage Ave.\nStudio City,..."
7651,7652,10-5-20 Smith Attachment.pdf,"Comments Received After September 30, 2020",1,0.000008,"COMMUNISM\nITS IDEOLOGY, ITS HISTORY,\nAND ITS..."
7707,7708,10-12-20 Haufrect.pdf,"Comments Received After September 30, 2020",1,0.000008,Public Input Template–2020 Ethnic Studies Mode...


In [138]:
matched_rows['submissions'].sum()

2082

In [70]:
matched_rows[matched_rows['text'].str.contains('1948', case=False, na=False)]

,Unnamed: 0,file_name,folder_name,submissions,weight,text
41,41,11-12-20 Galal.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: KGalal Sent: Thursday, November 12, 2020..."
108,108,11-13-20 Lavie.pdf,"Comments Received After September 30, 2020",1,0.000008,"Smadar Lavie, PhD Professor Emerita of Anthrop..."
114,114,11-13-20 Shoman Attachment.pdf,"Comments Received After September 30, 2020",1,0.000008,Arab Americans\n\nSample Lesson: Understanding...
117,117,11-13-20 Gutierrez.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Andrew Paul Gutierrez Sent: Friday, Nove..."
211,211,11-16-20 Dibas.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Nada Dibas Sent: Friday, November 13, 20..."
...,...,...,...,...,...,...
7619,7620,7-29-20 Institute for Curriculum Services.pdf,Comments Received After Field Review,1,0.000017,Jewish American Studies Course Outline:\nThe J...
7622,7623,Pietruszka.pdf,Comments Received After Field Review,1,0.000017,Dear CA Department of Education Ethnic Studies...
7651,7652,10-5-20 Smith Attachment.pdf,"Comments Received After September 30, 2020",1,0.000008,"COMMUNISM\nITS IDEOLOGY, ITS HISTORY,\nAND ITS..."
7707,7708,10-12-20 Haufrect.pdf,"Comments Received After September 30, 2020",1,0.000008,Public Input Template–2020 Ethnic Studies Mode...


In [139]:
nakba_variants = [
    'nakba',
    'Nakbah',
    'War of Independence',
    'Arab–Israeli War'
]

pattern = '|'.join(nakba_variants)

matched_rows = df[df['text'].str.contains(pattern, case=False, na=False)]
matched_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
20,20,11-18-20 Smith.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Warren Smith Sent: Wednesday, November 1..."
79,79,11-12-20 Lepawsky.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Jed Linden Sent: Thursday, November 12, ..."
108,108,11-13-20 Lavie.pdf,"Comments Received After September 30, 2020",1,0.000008,"Smadar Lavie, PhD Professor Emerita of Anthrop..."
114,114,11-13-20 Shoman Attachment.pdf,"Comments Received After September 30, 2020",1,0.000008,Arab Americans\n\nSample Lesson: Understanding...
131,131,11-12-20 Koatz.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: B Koatz Sent: Thursday, November 12, 202..."
...,...,...,...,...,...,...
7466,7467,7-27-20 Shapiro.pdf,Comments Received After Field Review,1,0.000017,"From: Larry Shapiro\nSent: Sunday, July 26, 20..."
7549,7550,7-13-20 Abdulhadi et al.pdf,Comments Received After Field Review,1,0.000017,CALIFORNIA SCHOLARS FOR ACADEMIC FREEDOM\nhttp...
7597,7598,1-13-20 Fouda.pdf,Comments Received After Field Review,1,0.000017,"From: MARGARET FOUDA\nSent: Sunday, January 12..."
7605,7606,8-26-19 Breslauer.pdf,Comments Received After Field Review,1,0.000017,"genJJ breskuer\n\nAugust 19, 2019\nMs. Soomin ..."


In [140]:
matched_rows['submissions'].sum()

334

# Duplicate Comments

In [81]:
duplicates = df[df['text'].str.contains('Education received', case=False)]

In [86]:
duplicates

,Unnamed: 0,file_name,folder_name,submissions,weight,text
30,30,11-12-20 Group Letter Critical Race Theory.pdf,"Comments Received After September 30, 2020",5,0.000040,The California Department of Education receive...
42,42,11-12-20 Group Letter Protect Butler.pdf,"Comments Received After September 30, 2020",3100,0.024743,The California Department of Education receive...
160,160,11-11-20 Group Letter Arab Americans 2.pdf,"Comments Received After September 30, 2020",4600,0.036716,The California Department of Education receive...
224,224,11-11-20 Group Letter Remove Political Agenda.pdf,"Comments Received After September 30, 2020",273,0.002179,The California Department of Education receive...
262,262,11-11-20 Group Letter Guiding Principles.pdf,"Comments Received After September 30, 2020",271,0.002163,The California Department of Education receive...
...,...,...,...,...,...,...
7056,7057,7-8-20 Group Letter Arab American Studies.pdf,Comments Received After Field Review,10000,0.173633,The California Department of Education receive...
7180,7181,7-24-20 Group Letter Protest Draft.pdf,Comments Received After Field Review,200,0.003473,The California Department of Education receive...
7455,7456,1-29-20 Group Email Arab American Studies.pdf,Comments Received After Field Review,900,0.015627,The California Department of Education receive...
7477,7478,7-8-20 Group Letter anti-BDS.pdf,Comments Received After Field Review,8,0.000139,The California Department of Education receive...


In [101]:
duplicates.to_csv('duplicates.csv', index=False)

In [100]:
df['submissions'].sum()

74466

# Pdfs of Only Duplicate Comments 

In [91]:
import os

copies_folder = os.path.join("/Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs")

valid_files_lower = set(f.strip().lower() for f in duplicates['file_name'].unique())

# recursively go through all subfolders
for root, dirs, files in os.walk(copies_folder):
    for file in files:
        file_path = os.path.join(root, file)
        # delete file if it's not in the dataframe
        if file.lower() not in valid_files_lower:
            os.remove(file_path)
            print("Deleted:", file_path)

Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/.DS_Store
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/10-26-20 Parker.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/10-1-20 Jesman.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/10-26-20 Friedberg.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/10-5-20 Smith Attachment.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text

Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/Comments received after 11-6-20 posting/11-13-20 Chang Attachment 1.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/Comments received after 11-6-20 posting/11-13-20 Sarfeh.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/Comments received after 11-6-20 posting/11-17-20 Dolan.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/Comments received after 11-6-20 posting/11-18-20 Nelson.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPRe

Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/Comments received after 11-6-20 posting/11-18-20 Yadegar.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/Comments received after 11-6-20 posting/11-16-20 Kava 2.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/Comments received after 11-6-20 posting/11-12-20 Mulford.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After September 30, 2020/Comments received after 11-6-20 posting/.DS_Store
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/T

Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/5-27-20 Villegas.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/7-24-20 Blumenthal.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/5-27-20 Rose.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/7-14-20 Nammar.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/9-12-19 Muwwakkil.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/T

Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/Lahav.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/7-30-20 Offerman.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/3-30-20 V Sandhu.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/5-26-20 Gallegos-Diaz.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/Dayan.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text

Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/Comments received after 7-31-20 posting/8-10-20 Gonzalez.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/Comments received after 7-31-20 posting/8-3-20 Landau.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/Comments received after 7-31-20 posting/8-11-20 Windham.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Comments Received After Field Review/Comments received after 7-31-20 posting/8-13-20 Thew.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_

Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/First Field Review (June - August 2019)/First Field Review (June - August 2019)/8-11-19 Oudiz.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/First Field Review (June - August 2019)/First Field Review (June - August 2019)/8-8-19 Feldman.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/First Field Review (June - August 2019)/First Field Review (June - August 2019)/8-9-19 Leiner.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/First Field Review (June - August 2019)/First Field Review (June - August 2019)/8-5-19 Ballard.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/

Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/First Field Review (June - August 2019)/First Field Review (June - August 2019)/7-30-19 Burke.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/First Field Review (June - August 2019)/First Field Review (June - August 2019)/8-12-19 DeMario docx.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/First Field Review (June - August 2019)/First Field Review (June - August 2019)/8-13-19 Pierluissi.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/First Field Review (June - August 2019)/First Field Review (June - August 2019)/8-15-19 Yang.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Te

Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Third Field Review (Dec 2020 - Jan 2021)/1-21-21 Burns et al.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Third Field Review (Dec 2020 - Jan 2021)/1-20-21 Banayan.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Third Field Review (Dec 2020 - Jan 2021)/1-21-21 Castroll.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Third Field Review (Dec 2020 - Jan 2021)/12-25-20 Kuchinsky.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/ethnicStudiesNLPResearch/Text Files/Text Files/duplicate_handling/duplicate_pdfs/Third Field Review (Dec 2020 - Jan 2021)/12-21-20 Sun.pdf
Deleted: /Users/maryamzakiyya/Desktop/NLP URAP/eth

In [92]:
pdf_count = 0

for root, dirs, files in os.walk(copies_folder):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_count += 1

print(f"Total PDF files: {pdf_count}")

Total PDF files: 91


# Illegible Texts Inspection

In [97]:
print(df.iloc[6331]['text'])

August 22, 2020

By electronic delivery only

Members of the Instructional Quality Commission California Department of Education 1430 N Street Sacramento, CA 95814-5901 ethnicstudies@cde.ca.gov

RE: Revised Ethnic Studies Model Curriculum

Dear Members of the Instructional Quality Commission:

I am a parent of three children who have gone through the public education system in California. My youngest son is currently a high school senior and is in the process of applying for college. I am writing to comment on the revised Ethnic Studies Model Curriculum (ESMC) released on July 31. My family and I wholeheartedly endorse the teaching of Ethnic Studies and we support Assembly Bill 331. Adding Ethnic Studies to California's high school curriculum will, among other things, foster respect between students and an appreciation for our state and country's rich and diverse population.

I wish to express my support for the removal of overt antisemitism and extreme and gratuitous anti-Israel bias 

# Duplicates Breakdown

In [102]:
dupe_antisemitism = pd.read_csv('duplicates _updated.csv')
dupe_antisemitism

,Unnamed: 0,file_name,folder_name,submissions,weight,text,jew_antisemitism
0,30,11-12-20 Group Letter Critical Race Theory.pdf,"Comments Received After September 30, 2020",5,0.000040,The California Department of Education receive...,0
1,42,11-12-20 Group Letter Protect Butler.pdf,"Comments Received After September 30, 2020",3100,0.024743,The California Department of Education receive...,1
2,160,11-11-20 Group Letter Arab Americans 2.pdf,"Comments Received After September 30, 2020",4600,0.036716,The California Department of Education receive...,0
3,224,11-11-20 Group Letter Remove Political Agenda.pdf,"Comments Received After September 30, 2020",273,0.002179,The California Department of Education receive...,0
4,262,11-11-20 Group Letter Guiding Principles.pdf,"Comments Received After September 30, 2020",271,0.002163,The California Department of Education receive...,0
...,...,...,...,...,...,...,...
86,7057,7-8-20 Group Letter Arab American Studies.pdf,Comments Received After Field Review,10000,0.173633,The California Department of Education receive...,0
87,7181,7-24-20 Group Letter Protest Draft.pdf,Comments Received After Field Review,200,0.003473,The California Department of Education receive...,1
88,7456,1-29-20 Group Email Arab American Studies.pdf,Comments Received After Field Review,900,0.015627,The California Department of Education receive...,0
89,7478,7-8-20 Group Letter anti-BDS.pdf,Comments Received After Field Review,8,0.000139,The California Department of Education receive...,1


In [104]:
dupe_antisemitism[dupe_antisemitism['jew_antisemitism'] == 1]['submissions'].sum()

37105

In [1]:
import pandas as pd

/Users/maryamzakiyya/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('all_files.csv')
df

,Unnamed: 0,file_name,folder_name,submissions,weight,text
0,0,11-13-20 Khan.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Hareem Khan Sent: Thursday, November 12,..."
1,1,11-13-20 Solkovits.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: gs <gsolk@aol.com> Sent: Thursday, Novem..."
2,2,11-13-20 Kaur and Singh.pdf,"Comments Received After September 30, 2020",1,0.000008,"November 13th, 2020\n\nDear Superintendent Thu..."
3,3,11-19-20 Lamont.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sami Lamont Sent: Wednesday, November 18..."
4,4,11-18-20 Jensen.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sine Hwang Jensen Sent: Tuesday, Novembe..."
...,...,...,...,...,...,...
7712,7713,10-27-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."
7714,7715,10-22-20 Nash.pdf,"Comments Received After September 30, 2020",1,0.000008,"Amin Nash, M.A. 10.21.2020\n\n1\n\nTo Executiv..."
7715,7716,10-27-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 27, ..."


In [3]:
df['folder_name'].unique()

array(['Comments Received After September 30, 2020',
       'Third Field Review (Dec 2020 - Jan 2021)',
       'Second Field Review (Sept 2020)',
       'Comments Received After Field Review',
       'Comments received after Third Field Review (after 1-21-21)',
       'First Field Review (June - August 2019)'], dtype=object)

In [7]:
df[df['folder_name'].str.contains('First Field Review', case=False)]

,Unnamed: 0,file_name,folder_name,submissions,weight,text
1736,1736,8-7-19 Peters.pdf,First Field Review (June - August 2019),1,0.000018,"From: Lorin Peters Sent: Tuesday, August 6, 20..."
1737,1737,8-7-19 Pastcan.pdf,First Field Review (June - August 2019),1,0.000018,Public Input Template�2020 Ethnic Studies Mode...
1738,1738,8-13-19 Friedman.pdf,First Field Review (June - August 2019),1,0.000018,"From: Mona S Marley Sent: Tuesday, August 13, ..."
1739,1739,8-9-19 Kronick.pdf,First Field Review (June - August 2019),1,0.000018,"From: Mel Kronick Sent: Friday, August 9, 2019..."
1740,1740,8-8-19 Vosicher.pdf,First Field Review (June - August 2019),1,0.000018,"From: Sally Vosicher Sent: Thursday, August 8,..."
...,...,...,...,...,...,...
5866,5866,8-9-19 Gale.pdf,First Field Review (June - August 2019),1,0.000018,"258 A St, Ste.1, PMB 118\nAshland, OR 97520 ki..."
5867,5867,8-9-19 Hiatt.pdf,First Field Review (June - August 2019),1,0.000018,Public Input Template-2020 Ethnic Studies Mode...
5868,5868,8-13-19 Levi.pdf,First Field Review (June - August 2019),1,0.000018,﻿8/9/2019\nSuperintendent Tony Thurmond\nState...
5869,5869,8-9-19 Cohon.pdf,First Field Review (June - August 2019),1,0.000018,Regarding: California Ethnic Studies Model Cur...
